<a href="https://colab.research.google.com/github/MaryamAlhosani1/TPCDS_Parquet_ORC/blob/main/TPCDS_Parquet_ORC_Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parquet vs. ORC on TPC-DS

This notebook analyzes saved benchmark measurements for five TPC-DS tables (`customer`, `item`, `store`, `date_dim`, and `store_sales`) at scale factors 1, 2, and 5. We evaluate storage efficiency, per-table compression ratios, write performance, and row-count query execution times. Additionally, we examine two scale factor 1 (SF1) experiments addressing compression behavior across different data types and the format sensitivity when a column is removed.

Our analysis and comparisons are contextualized against the findings of Liu et al. (2025): *An Empirical Evaluation of Columnar File Formats: Parquet vs. ORC on TPC-DS*.

## 1. Colab setup

We first clone the benchmark repository, set the working directory, and install the required Python and PySpark dependencies.

In [14]:
from pathlib import Path
import subprocess

repo_dir = Path("/content/TPCDS_Parquet_ORC")
repo_url = "https://github.com/MaryamAlhosani1/TPCDS_Parquet_ORC.git"

if not (repo_dir / ".git").is_dir():
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)

%cd /content/TPCDS_Parquet_ORC

/content/TPCDS_Parquet_ORC


In [15]:
%pip install -r requirements.txt

## 2. Regenerate summaries and figures

We execute the analysis-only suite to process the raw repository measurement CSVs and regenerate the summary tables and figures.

In [16]:
import sys

subprocess.run(
    [sys.executable, "run_all.py", "--analysis-only"],
    cwd=repo_dir,
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', 'run_all.py', '--analysis-only'], returncode=0)

## 3. Compression across scale factors

This section analyzes the file size and compression ratios. A lower compressed-to-raw ratio represents superior storage efficiency.

We begin by loading and previewing the raw SF1 compression results to verify the structure of the data.

In [6]:
import pandas as pd

sf1 = pd.read_csv(
    "/content/TPCDS_Parquet_ORC/results/compression_sf1/compression_results.csv"
)

display(sf1[["table", "parquet_ratio", "orc_ratio", "winner"]].round(3))

,table,parquet_ratio,orc_ratio,winner
0,customer,0.424,0.409,ORC
1,item,0.356,0.341,ORC
2,store,3.313,1.725,ORC
3,date_dim,0.183,0.049,ORC
4,store_sales,0.380,0.444,Parquet


## 4. Write and row-count timings

We compare the write execution times and row-count query performance for both formats. Measurements are recorded in seconds.

In [27]:
import pandas as pd
from pathlib import Path

# Ensure we read the file correctly using the verified absolute path
stats_dir = Path("/content/TPCDS_Parquet_ORC/results/statistics")

# Fallback to general compression_summary.csv if present, or build from results if missing
summary_file = stats_dir / "compression_summary.csv"
if not summary_file.exists():
    # Fallback/construction mechanism from separate scale factor CSVs if the summary file was not generated
    frames = []
    for sf in [1, 2, 5]:
        sf_file = Path(f"/content/TPCDS_Parquet_ORC/results/compression_sf{sf}/compression_results.csv")
        if sf_file.exists():
            df = pd.read_csv(sf_file)
            df["scale_factor"] = sf
            frames.append(df)
    compression = pd.concat(frames, ignore_index=True)
else:
    compression = pd.read_csv(summary_file)

size_columns = ["raw_bytes", "parquet_directory_bytes", "orc_directory_bytes"]
totals = compression.groupby("scale_factor")[size_columns].sum()

comparison = pd.DataFrame({
    "Parquet (% of raw size)": 100 * totals["parquet_directory_bytes"] / totals["raw_bytes"],
    "ORC (% of raw size)": 100 * totals["orc_directory_bytes"] / totals["raw_bytes"],
})

display(comparison.round(2))

,Parquet (% of raw size),ORC (% of raw size)
scale_factor,,
1,37.64,43.17
2,38.07,43.45
5,37.85,44.16


In [19]:
# Map individual table winners across all scale factors
table_winners = compression.pivot(
    index="table",
    columns="scale_factor",
    values="winner",
).rename(columns=lambda scale: f"SF{scale}")

display(table_winners)

scale_factor,SF1,SF2,SF5
table,,,
customer,ORC,ORC,ORC
date_dim,ORC,ORC,ORC
item,ORC,ORC,ORC
store,ORC,ORC,ORC
store_sales,Parquet,Parquet,Parquet


### Findings
* **Aggregate Footprint:** Across SF1, SF2, and SF5, Parquet consistently consumed less total storage than ORC (approximately 38% vs. 43% of raw bytes).
* **Per-Table Winners:** ORC produced smaller file directories for four out of five tables (`customer`, `date_dim`, `item`, and `store`) at every scale. However, Parquet achieved significantly better compression on the massive `store_sales` table, dictating the overall aggregate storage winner.

In [10]:
from pathlib import Path
import pandas as pd

stats_dir = Path("/content/TPCDS_Parquet_ORC/results/statistics")
timing = pd.read_csv(stats_dir / "performance_summary.csv")

display(timing[[
    "scale_factor",
    "parquet_total_write_sec",
    "orc_total_write_sec",
    "parquet_total_count_sec",
    "orc_total_count_sec",
]].round(2))

,scale_factor,parquet_total_write_sec,orc_total_write_sec,parquet_total_count_sec,orc_total_count_sec
0,1,33.07,27.27,1.96,1.57
1,2,54.72,57.05,1.98,1.88
2,5,139.96,125.40,1.75,1.19


### Findings
* **Write Performance:** ORC completed writes faster at SF1 (27.27s vs. 33.07s) and SF5 (125.40s vs. 139.96s), while Parquet demonstrated superior write speed at SF2 (54.72s vs. 57.05s).
* **Row-Count Performance:** ORC completed the row-count queries faster than Parquet across all three scale factors.

In [20]:
types = pd.read_csv(stats_dir / "data_type_comparison.csv")

# Adjust display names for academic readability
display(types[[
    "data_type",
    "parquet_data_bytes",
    "orc_data_bytes",
    "observed_smaller_format",
    "paper_smaller_format",
]])

,data_type,parquet_data_bytes,orc_data_bytes,observed_smaller_format,paper_smaller_format
0,Date,296843,5737,ORC,Not separately reported
1,FloatingPoint,119386692,148346251,Parquet,Parquet
2,Integer,30687837,25027579,ORC,ORC
3,String,5352992,5235930,ORC,Parquet


## 5. Compression by data type

Here we evaluate how each format performs across different data types at SF1 and compare our observations with the findings published in the reference paper.

### Findings
* **Integers and Floating Points:** Our observed smaller formats are fully consistent with Liu et al. (2025). ORC is more space-efficient for integers, whereas Parquet is more space-efficient for floating-point values.
* **Strings:** Our local measurements show ORC to be slightly smaller for string columns, while the original study reported Parquet as smaller. This discrepancy might arise from minor configuration or environment variances.

In [21]:
sensitivity = pd.read_csv(stats_dir / "sensitivity_analysis.csv")

data_files = sensitivity.loc[
    sensitivity["measure"].eq("data_bytes"),
    ["format", "before_bytes", "after_bytes", "reduction_pct"],
]

display(data_files.round(4))

,format,before_bytes,after_bytes,reduction_pct
0,parquet,5553790,5553068,0.0130
2,orc,5367091,5366848,0.0045


## 6. Sensitivity to removing a column

This sensitivity test measures the percentage reduction in compressed data file sizes after removing the final column of the SF1 `customer` table.

### Findings
Removing the final column resulted in negligible size reductions for both formats: only **0.0130%** for Parquet and **0.0045%** for ORC. This indicates high structural resilience to singular column omissions in both columnar layouts.

## 7. Full benchmark reproduction

To perform a complete reproduction of the Apache Spark measurements across all tables and scale factors, the repository provides the following command:

```bash
python run_all.py
```

*Note on Trial Execution:* The full benchmark workflow was initiated below but manually interrupted during the SF2 dataset configuration stage after successfully generating and validating the SF1 compression measurements.

In [13]:
%cd /content/TPCDS_Parquet_ORC
!python run_all.py

/content/TPCDS_Parquet_ORC

/usr/bin/python3 -u /content/TPCDS_Parquet_ORC/src/sequence.py
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 06:13:45 WARN NativeCodeLoader:

## 8. Conclusion

In this notebook, we evaluated Parquet and ORC columnar formats across five TPC-DS benchmarks. Parquet demonstrated superior aggregate storage efficiency across SF1, SF2, and SF5, primarily due to its efficient compression of the massive `store_sales` table. Conversely, ORC achieved smaller footprints for the four smaller dimension and customer tables at every scale factor. For query speeds, ORC consistently outperformed Parquet on row-count operations, while write performances varied by scale factor.

Our data type analysis confirmed that ORC excels at integer compression and Parquet excels at floating-point values, aligning with the reference paper. A slight difference was observed in string compression, where ORC was smaller in our setup. Sensitivity analysis showed that removing a single column had negligible effects on total compressed storage for both formats.